# Phase 03D: ExpansionNet v2 Full Pseudo-Caption Generation

Notebook sinh pseudo-caption production cho toàn bộ **113.287 ảnh Karpathy
Train+Restval** bằng frozen Teacher A ExpansionNet v2.

## Contract đã khóa từ Phase 03C

- batch size: `8`;
- Beam-3, length penalty `0.7`;
- tối đa `31` decoder steps;
- image order tăng dần theo `image_id`;
- một shard duy nhất: `[0, 113287)`;
- output đồng thời giữ native Teacher IDs và S1 token IDs;
- checkpoint tiến độ định kỳ, hỗ trợ resume;
- JSONL hoàn chỉnh được đổi tên nguyên tử;
- tạo Parquet tối ưu cho Phase 4 Sequence KD;
- direct Logit KD bị cấm;
- Held-out Test luôn `SEALED`.

## Kaggle inputs

1. Phase 1 Formal Output.
2. Phase 2A Shared Student Pipeline.
3. Phase 03B ExpansionNet v2 output.
4. Phase 03C ExpansionNet v2 readiness output.
5. MS COCO 2014 images.
6. ExpansionNet v2 offline model.
7. Khi resume: gắn output Phase 03D trước đó có progress marker.

GPU T4 hoặc mạnh hơn, Internet Off. Thời gian ước tính khoảng 3–4 giờ.


## Cell 1 — Authorization, integrity và production contract

Xác minh Phase 03C `11/11` artifact, một shard, batch 8 và quyền chạy Phase
03D. Chưa đọc ảnh hay checkpoint.


In [1]:
import csv
import gc
import hashlib
import json
import math
import os
import pickle
import platform
import random
import re
import shutil
import sys
import time
import unicodedata
from argparse import Namespace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 42
INPUT_ROOT = Path("/kaggle/input")
NOTEBOOK_INPUT_ROOT = INPUT_ROOT / "notebooks"
MODEL_INPUT_ROOT = INPUT_ROOT / "models"
DATASET_INPUT_ROOT = INPUT_ROOT / "datasets"
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase3_teacher_a_expansionnet_v2_pseudocaptions"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
PSEUDOCAPTION_DIR = OUTPUT_ROOT / "pseudocaptions"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR,
    PSEUDOCAPTION_DIR, MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one_under(root, name):
    matches = sorted(Path(root).rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {root}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA không khả dụng. Phase 03D bắt buộc GPU."
    )
set_deterministic(SEED)
device = torch.device("cuda:0")

PHASE3C_READY_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT,
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_"
    "PSEUDOCAPTION_READINESS_READY.json",
)
PHASE3C_ROOT = PHASE3C_READY_PATH.parent
PHASE3B_READY_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT,
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json",
)
PHASE3B_ROOT = PHASE3B_READY_PATH.parent
PHASE1_READY_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT, "PHASE1_FORMAL_DELIVERABLES_READY.json"
)
PHASE2_READY_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT, "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
phase3c_ready = read_json(PHASE3C_READY_PATH)
phase3b_ready = read_json(PHASE3B_READY_PATH)
assert phase3c_ready["status"] == "ready"
assert phase3c_ready["teacher"] == "ExpansionNet v2"
assert phase3c_ready["matched_student"] == "S1 MobileNetV3-Large"
assert phase3c_ready["selected_batch_size"] == 8
assert phase3c_ready["recommended_shards"] == 1
assert phase3c_ready["sequence_kd_ready"] is True
assert phase3c_ready[
    "phase03d_full_pseudocaption_generation_authorized"
] is True
assert phase3c_ready["test_set_status"] == "sealed"
assert phase3c_ready["test_set_used"] is False

manifest_ref = phase3c_ready["manifest"]["csv"]
sums_ref = phase3c_ready["manifest"]["sha256sums"]
PHASE3C_MANIFEST = resolve_inside(
    PHASE3C_ROOT, manifest_ref["relative_path"]
)
PHASE3C_SUMS = resolve_inside(
    PHASE3C_ROOT, sums_ref["relative_path"]
)
assert sha256_file(PHASE3C_MANIFEST) == manifest_ref["sha256"]
assert sha256_file(PHASE3C_SUMS) == sums_ref["sha256"]
with PHASE3C_MANIFEST.open(
    "r", encoding="utf-8", newline=""
) as handle:
    phase3c_rows = list(csv.DictReader(handle))
assert len(phase3c_rows) == 11
for row in phase3c_rows:
    artifact = resolve_inside(PHASE3C_ROOT, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

for key in [
    "production_contract", "token_mapping_contract", "shard_plan"
]:
    reference = phase3c_ready[key]
    artifact = resolve_inside(
        PHASE3C_ROOT, reference["relative_path"]
    )
    assert sha256_file(artifact) == reference["sha256"]

PRODUCTION_CONTRACT_PATH = resolve_inside(
    PHASE3C_ROOT,
    phase3c_ready["production_contract"]["relative_path"],
)
TOKEN_MAPPING_PATH = resolve_inside(
    PHASE3C_ROOT,
    phase3c_ready["token_mapping_contract"]["relative_path"],
)
SHARD_PLAN_PATH = resolve_inside(
    PHASE3C_ROOT,
    phase3c_ready["shard_plan"]["relative_path"],
)
production_contract = read_json(PRODUCTION_CONTRACT_PATH)
token_mapping_contract = read_json(TOKEN_MAPPING_PATH)
shard_plan = read_json(SHARD_PLAN_PATH)
assert production_contract["total_images"] == 113287
assert production_contract["generation"]["selected_batch_size"] == 8
assert shard_plan["recommended_shards"] == 1
assert shard_plan["shards"][0]["start_index"] == 0
assert shard_plan["shards"][0]["stop_index_exclusive"] == 113287

write_json(
    AUDIT_DIR / "phase3d_cell01_authorization.json",
    {
        "audit_version": "phase3d_expansionnet_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "phase3c_manifest_rows": len(phase3c_rows),
        "selected_batch_size": 8,
        "authorized_shards": 1,
        "authorized_images": 113287,
        "full_generation_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03D EXPANSIONNET V2 CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 03C manifest        : 11 / 11 VERIFIED")
print("Production contract       : VERIFIED")
print("Token mapping contract    : VERIFIED")
print("Authorized images         : 113287")
print("Authorized shards         : 1")
print("Batch size                : 8")
print("Full generation started   : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 6 code cells")
print("Next                      : image index and tokenizer")


PHASE 03D EXPANSIONNET V2 CELL 1 STATUS
Status                    : PASSED
Phase 03C manifest        : 11 / 11 VERIFIED
Production contract       : VERIFIED
Token mapping contract    : VERIFIED
Authorized images         : 113287
Authorized shards         : 1
Batch size                : 8
Full generation started   : False
Test Set status           : SEALED
Progress                  : 1 / 6 code cells
Next                      : image index and tokenizer


## Cell 2 — Train+Restval index và locked S1 tokenizer

Dựng lại đúng image order đã khóa, không đọc training caption. Chuẩn bị hàm
chuyển caption của Teacher thành chuỗi token S1.


In [2]:
CELL_STARTED_AT = time.perf_counter()
import pyarrow as pa
import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

IMAGE_MANIFEST_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT, "mscoco_image_manifest.parquet"
)
VOCAB_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT, "word_vocabulary_minfreq5.json"
)
TOKENIZER_PATH = exactly_one_under(
    NOTEBOOK_INPUT_ROOT, "tokenizer_contract.json"
)
vocabulary = read_json(VOCAB_PATH)
tokenizer_contract = read_json(TOKENIZER_PATH)
TOKEN_TO_ID = {
    token: index
    for index, token in enumerate(vocabulary["id_to_token"])
}
TOKEN_PATTERN = re.compile(tokenizer_contract["token_pattern"])
assert len(TOKEN_TO_ID) == 10080


def phase1_tokenize(text):
    normalized = unicodedata.normalize("NFKC", text).lower()
    normalized = (
        normalized.replace("’", "'")
        .replace("‘", "'")
        .replace("`", "'")
    )
    return TOKEN_PATTERN.findall(normalized)


def caption_to_s1(caption):
    lexical = phase1_tokenize(caption)[:30]
    token_ids = (
        [1] + [TOKEN_TO_ID.get(token, 3) for token in lexical] + [2]
    )
    assert 2 <= len(token_ids) <= 32
    return lexical, token_ids


def find_coco_leaf(relative_parts, probe_prefix):
    matches = []
    for owner in DATASET_INPUT_ROOT.iterdir():
        candidate = owner.joinpath(*relative_parts)
        if (
            candidate.is_dir()
            and next(candidate.glob(f"{probe_prefix}*.jpg"), None)
        ):
            matches.append(candidate.resolve())
    assert len(matches) == 1, matches
    return matches[0]


TRAIN_LEAF = find_coco_leaf(
    ("mscoco2014-1", "train2014", "train2014"),
    "COCO_train2014_",
)
VAL_LEAF = find_coco_leaf(
    ("mscoco2014-1", "val2014", "val2014"),
    "COCO_val2014_",
)
manifest_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=[
        "image_id", "experiment_split", "source_partition", "image_path"
    ],
)
TRAIN_PATHS = {}
VALIDATION_IDS = set()
TEST_IDS = set()
for row in manifest_table.to_pylist():
    image_id = int(row["image_id"])
    split = row["experiment_split"]
    if split == "held_out_test":
        TEST_IDS.add(image_id)
    elif split == "validation":
        VALIDATION_IDS.add(image_id)
    elif split == "train":
        leaf = (
            TRAIN_LEAF
            if row["source_partition"] == "train2014"
            else VAL_LEAF
        )
        TRAIN_PATHS[image_id] = leaf / Path(row["image_path"]).name
assert len(TRAIN_PATHS) == 113287
assert len(VALIDATION_IDS) == 5000
assert len(TEST_IDS) == 5000
assert set(TRAIN_PATHS).isdisjoint(VALIDATION_IDS | TEST_IDS)
TRAIN_IMAGE_IDS = sorted(TRAIN_PATHS)
assert TRAIN_IMAGE_IDS[0] == shard_plan["shards"][0]["first_image_id"]
assert TRAIN_IMAGE_IDS[-1] == shard_plan["shards"][0]["last_image_id"]

PREPROCESS = transforms.Compose(
    [
        transforms.Resize(
            (384, 384), interpolation=InterpolationMode.BILINEAR
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


class TrainImages(Dataset):
    def __init__(self, start_index=0):
        self.start_index = int(start_index)

    def __len__(self):
        return len(TRAIN_IMAGE_IDS) - self.start_index

    def __getitem__(self, relative_index):
        index = self.start_index + relative_index
        image_id = TRAIN_IMAGE_IDS[index]
        with Image.open(TRAIN_PATHS[image_id]) as source:
            image = PREPROCESS(source.convert("RGB"))
        return {
            "global_index": index,
            "image_id": image_id,
            "image": image,
        }


write_json(
    AUDIT_DIR / "phase3d_cell02_data_and_tokenizer.json",
    {
        "audit_version": "phase3d_expansionnet_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "train_restval_images": 113287,
        "validation_ids": 5000,
        "held_out_test_ids": 5000,
        "membership_overlaps": 0,
        "image_order": "ascending image_id",
        "student_vocabulary_size": 10080,
        "maximum_lexical_tokens": 30,
        "training_caption_read": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03D EXPANSIONNET V2 CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Train+Restval images      : 113287")
print("Image order               : ascending image_id")
print("Student vocabulary        : 10080")
print("Maximum lexical tokens    : 30")
print("Membership overlaps       : 0")
print("Training caption read     : False")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 6 code cells")
print("Next                      : strict-load Teacher and resume discovery")


PHASE 03D EXPANSIONNET V2 CELL 2 STATUS
Status                    : PASSED
Train+Restval images      : 113287
Image order               : ascending image_id
Student vocabulary        : 10080
Maximum lexical tokens    : 30
Membership overlaps       : 0
Training caption read     : False
Test Set status           : SEALED
Progress                  : 2 / 6 code cells
Next                      : strict-load Teacher and resume discovery


## Cell 3 — Frozen Teacher strict-load và resume discovery

Nếu output Phase 03D cũ được gắn làm input, notebook chọn progress marker có
nhiều record hoàn chỉnh nhất, xác minh SHA-256 rồi tiếp tục từ batch boundary.


In [3]:
CELL_STARTED_AT = time.perf_counter()

model_manifests = []
for candidate in MODEL_INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        payload = read_json(candidate)
    except Exception:
        continue
    if payload.get("model_id") == "expansionnet_v2":
        model_manifests.append((candidate.resolve(), payload))
assert len(model_manifests) == 1, model_manifests
MODEL_MANIFEST_PATH, model_manifest = model_manifests[0]
MODEL_ROOT = MODEL_MANIFEST_PATH.parent
CHECKPOINT_PATH = (
    MODEL_ROOT / model_manifest["checkpoint"]["relative_path"]
)
assert CHECKPOINT_PATH.stat().st_size == int(
    model_manifest["checkpoint"]["size_bytes"]
)
assert model_manifest["checkpoint"]["sha256"] == (
    phase3b_ready["checkpoint_sha256"]
)
NATIVE_VOCAB_PATH = (
    MODEL_ROOT / model_manifest["vocabulary"]["relative_path"]
)
with NATIVE_VOCAB_PATH.open("rb") as handle:
    NATIVE_TOKENS = pickle.load(handle)
NATIVE_WORD2IDX = NATIVE_TOKENS["word2idx_dict"]
NATIVE_IDX2WORD = NATIVE_TOKENS["idx2word_list"]
NATIVE_BOS = int(NATIVE_WORD2IDX["SOS"])
NATIVE_EOS = int(NATIVE_WORD2IDX["EOS"])
NATIVE_PAD = int(NATIVE_WORD2IDX["PAD"])

SOURCE_ROOT = MODEL_ROOT / "ExpansionNet_v2_src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

drop_args = Namespace(
    enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0
)
TEACHER = End_ExpansionNet_v2(
    swin_img_size=384, swin_patch_size=4, swin_in_chans=3,
    swin_embed_dim=192, swin_depths=[2, 2, 18, 2],
    swin_num_heads=[6, 12, 24, 48], swin_window_size=12,
    swin_mlp_ratio=4.0, swin_qkv_bias=True, swin_qk_scale=None,
    swin_drop_rate=0.0, swin_attn_drop_rate=0.0,
    swin_drop_path_rate=0.0, swin_norm_layer=torch.nn.LayerNorm,
    swin_ape=False, swin_patch_norm=True, swin_use_checkpoint=False,
    final_swin_dim=1536, d_model=512, N_enc=3, N_dec=3,
    num_heads=8, ff=2048,
    num_exp_enc_list=[32, 64, 128, 256, 512],
    num_exp_dec=16, output_word2idx=NATIVE_WORD2IDX,
    output_idx2word=NATIVE_IDX2WORD, max_seq_len=74,
    drop_args=drop_args, rank=device,
)
checkpoint = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
incompatible = TEACHER.load_state_dict(
    checkpoint["model_state_dict"], strict=True
)
assert not incompatible.missing_keys
assert not incompatible.unexpected_keys
del checkpoint
gc.collect()
TEACHER = TEACHER.to(device).eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad_(False)


def decode_native(tokens):
    words = []
    for token in tokens:
        token = int(token)
        if token == NATIVE_BOS:
            continue
        if token == NATIVE_EOS:
            break
        if token == NATIVE_PAD:
            continue
        words.append(str(NATIVE_IDX2WORD[token]))
    return " ".join(words).strip()


def generate_batch(images):
    images = images.to(device, non_blocking=True)
    with torch.inference_mode():
        predictions, log_probs = TEACHER(
            enc_x=images,
            enc_x_num_pads=[0] * images.shape[0],
            mode="beam_search",
            beam_size=3,
            beam_max_seq_len=31,
            sample_or_max="max",
            how_many_outputs=1,
            sos_idx=NATIVE_BOS,
            eos_idx=NATIVE_EOS,
            length_penalty=0.7,
        )
    tokens = [
        [int(value) for value in predictions[index][0]]
        for index in range(len(predictions))
    ]
    captions = [decode_native(values) for values in tokens]
    return tokens, captions


PARTIAL_PATH = (
    PSEUDOCAPTION_DIR
    / "teacher_a_expansionnet_v2_train_restval.partial.jsonl"
)
FINAL_JSONL = (
    PSEUDOCAPTION_DIR
    / "teacher_a_expansionnet_v2_train_restval.jsonl"
)
PROGRESS_NAME = (
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_PSEUDOCAPTION_PROGRESS.json"
)
PROGRESS_PATH = OUTPUT_ROOT / PROGRESS_NAME
progress_candidates = []
for candidate in NOTEBOOK_INPUT_ROOT.rglob(PROGRESS_NAME):
    try:
        payload = read_json(candidate)
    except Exception:
        continue
    if (
        payload.get("teacher_checkpoint_sha256")
        == phase3b_ready["checkpoint_sha256"]
        and payload.get("total_records") == 113287
        and payload.get("status") == "in_progress"
    ):
        progress_candidates.append((candidate.resolve(), payload))

RESUME_USED = False
COMPLETED_RECORDS = 0
if progress_candidates:
    progress_candidates.sort(
        key=lambda item: int(item[1]["completed_records"])
    )
    previous_progress_path, previous_progress = (
        progress_candidates[-1]
    )
    previous_partial = resolve_inside(
        previous_progress_path.parent,
        previous_progress["partial_relative_path"],
    )
    assert previous_partial.is_file()
    assert previous_partial.stat().st_size == int(
        previous_progress["partial_size_bytes"]
    )
    assert sha256_file(previous_partial) == (
        previous_progress["partial_sha256"]
    )
    shutil.copy2(previous_partial, PARTIAL_PATH)
    expected_index = 0
    with PARTIAL_PATH.open("r", encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            assert record["image_id"] == TRAIN_IMAGE_IDS[expected_index]
            assert record["global_index"] == expected_index
            expected_index += 1
    COMPLETED_RECORDS = expected_index
    assert COMPLETED_RECORDS == int(
        previous_progress["completed_records"]
    )
    RESUME_USED = True

write_json(
    AUDIT_DIR / "phase3d_cell03_model_and_resume.json",
    {
        "audit_version": "phase3d_expansionnet_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "strict_load": True,
        "missing_keys": 0,
        "unexpected_keys": 0,
        "frozen_teacher": True,
        "resume_used": RESUME_USED,
        "resume_completed_records": COMPLETED_RECORDS,
        "resume_validation_exact": True,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03D EXPANSIONNET V2 CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Strict checkpoint load    : PASSED")
print("Frozen Teacher            : True")
print("Resume candidate used     :", RESUME_USED)
print("Completed records restored:", COMPLETED_RECORDS)
print("Resume validation exact   : True")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 6 code cells")
print("Next                      : full production generation")


PHASE 03D EXPANSIONNET V2 CELL 3 STATUS
Status                    : PASSED
Strict checkpoint load    : PASSED
Frozen Teacher            : True
Resume candidate used     : False
Completed records restored: 0
Resume validation exact   : True
Test Set status           : SEALED
Progress                  : 3 / 6 code cells
Next                      : full production generation


## Cell 4 — Full 113.287-image production generation

Ghi `.partial.jsonl`, flush/fsync và cập nhật progress marker sau mỗi 250
batch (2.000 ảnh). Khi hoàn tất, đổi tên nguyên tử sang JSONL chính thức.


In [4]:
CELL_STARTED_AT = time.perf_counter()
BATCH_SIZE = 8
CHECKPOINT_EVERY_BATCHES = 250
remaining_dataset = TrainImages(start_index=COMPLETED_RECORDS)
remaining_loader = DataLoader(
    remaining_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=(len(remaining_dataset) > 0),
)
mode = "a" if RESUME_USED else "w"
generation_started = time.perf_counter()
completed = COMPLETED_RECORDS
with PARTIAL_PATH.open(
    mode, encoding="utf-8", newline="\n"
) as output_handle:
    for batch_index, batch in enumerate(remaining_loader, start=1):
        teacher_ids, captions = generate_batch(batch["image"])
        image_ids = [int(value) for value in batch["image_id"]]
        global_indices = [
            int(value) for value in batch["global_index"]
        ]
        for global_index, image_id, native_ids, caption in zip(
            global_indices, image_ids, teacher_ids, captions
        ):
            assert global_index == completed
            assert image_id == TRAIN_IMAGE_IDS[global_index]
            assert caption
            lexical, student_ids = caption_to_s1(caption)
            record = {
                "global_index": global_index,
                "image_id": image_id,
                "caption": caption,
                "teacher_token_ids": native_ids,
                "student_token_ids": student_ids,
                "teacher_checkpoint_sha256": (
                    phase3b_ready["checkpoint_sha256"]
                ),
                "shard_id": 0,
            }
            output_handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )
            completed += 1

        should_checkpoint = (
            batch_index % CHECKPOINT_EVERY_BATCHES == 0
            or completed == 113287
        )
        if should_checkpoint:
            output_handle.flush()
            os.fsync(output_handle.fileno())
            partial_sha = sha256_file(PARTIAL_PATH)
            elapsed = time.perf_counter() - generation_started
            processed_this_run = completed - COMPLETED_RECORDS
            rate = processed_this_run / max(elapsed, 1e-9)
            remaining_seconds = (
                (113287 - completed) / max(rate, 1e-9)
            )
            write_json(
                PROGRESS_PATH,
                {
                    "schema_version": (
                        "phase3d_pseudocaption_progress_v1"
                    ),
                    "created_at_utc": now_utc(),
                    "status": "in_progress",
                    "teacher": "ExpansionNet v2",
                    "teacher_checkpoint_sha256": (
                        phase3b_ready["checkpoint_sha256"]
                    ),
                    "total_records": 113287,
                    "completed_records": completed,
                    "last_completed_global_index": completed - 1,
                    "last_completed_image_id": (
                        TRAIN_IMAGE_IDS[completed - 1]
                    ),
                    "partial_relative_path": (
                        PARTIAL_PATH.relative_to(
                            OUTPUT_ROOT
                        ).as_posix()
                    ),
                    "partial_size_bytes": PARTIAL_PATH.stat().st_size,
                    "partial_sha256": partial_sha,
                    "batch_size": BATCH_SIZE,
                    "resume_used": RESUME_USED,
                    "elapsed_this_run_seconds": elapsed,
                    "images_per_second": rate,
                    "estimated_remaining_seconds": remaining_seconds,
                    "test_set_status": "sealed",
                    "test_set_used": False,
                },
            )
            print(
                f"Production: {completed}/113287 | "
                f"{rate:.2f} image/s | "
                f"ETA={remaining_seconds / 3600:.2f}h",
                flush=True,
            )

assert completed == 113287
assert PARTIAL_PATH.is_file()
os.replace(PARTIAL_PATH, FINAL_JSONL)
final_jsonl_sha = sha256_file(FINAL_JSONL)
write_json(
    PROGRESS_PATH,
    {
        "schema_version": "phase3d_pseudocaption_progress_v1",
        "created_at_utc": now_utc(),
        "status": "generation_complete",
        "teacher": "ExpansionNet v2",
        "teacher_checkpoint_sha256": (
            phase3b_ready["checkpoint_sha256"]
        ),
        "total_records": 113287,
        "completed_records": 113287,
        "final_relative_path": FINAL_JSONL.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "final_size_bytes": FINAL_JSONL.stat().st_size,
        "final_sha256": final_jsonl_sha,
        "batch_size": BATCH_SIZE,
        "resume_used": RESUME_USED,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
write_json(
    AUDIT_DIR / "phase3d_cell04_full_generation.json",
    {
        "audit_version": "phase3d_expansionnet_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "records": 113287,
        "batch_size": BATCH_SIZE,
        "resume_used": RESUME_USED,
        "starting_record": COMPLETED_RECORDS,
        "atomic_final_rename": True,
        "final_jsonl_sha256": final_jsonl_sha,
        "full_generation_complete": True,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03D EXPANSIONNET V2 CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Production records        : 113287 / 113287")
print("Batch size                :", BATCH_SIZE)
print("Resume used               :", RESUME_USED)
print("Atomic final rename       : PASSED")
print("Final JSONL               :", FINAL_JSONL)
print("Final JSONL SHA-256       :", final_jsonl_sha)
print("Test Set status           : SEALED")
print("Progress                  : 4 / 6 code cells")
print("Next                      : validate and export Parquet")


/kaggle/input/models/nguynthnhdan/expansionnet-v2-ieee-bigdata2023/pytorch/default/1/expansionnet_v2_ieee_bigdata2023/ExpansionNet_v2_src/models/captioning_model.py:206: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  new_loop_dec_classes = loop_dec_classes[[bs_idxes, which_sequence]]
/kaggle/input/models/nguynthnhdan/expansionnet-v2-ieee-bigdata2023/pytorch/default/1/expansionnet_v2_ieee_bigdata2023/ExpansionNet_v2_src/models/captioning_model.py:207: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor

Production: 2000/113287 | 10.01 image/s | ETA=3.09h
Production: 4000/113287 | 9.81 image/s | ETA=3.09h
Production: 6000/113287 | 9.75 image/s | ETA=3.06h
Production: 8000/113287 | 9.72 image/s | ETA=3.01h
Production: 10000/113287 | 9.70 image/s | ETA=2.96h
Production: 12000/113287 | 9.69 image/s | ETA=2.90h
Production: 14000/113287 | 9.68 image/s | ETA=2.85h
Production: 16000/113287 | 9.68 image/s | ETA=2.79h
Production: 18000/113287 | 9.67 image/s | ETA=2.74h
Production: 20000/113287 | 9.66 image/s | ETA=2.68h
Production: 22000/113287 | 9.66 image/s | ETA=2.62h
Production: 24000/113287 | 9.66 image/s | ETA=2.57h
Production: 26000/113287 | 9.65 image/s | ETA=2.51h
Production: 28000/113287 | 9.65 image/s | ETA=2.45h
Production: 30000/113287 | 9.65 image/s | ETA=2.40h
Production: 32000/113287 | 9.65 image/s | ETA=2.34h
Production: 34000/113287 | 9.65 image/s | ETA=2.28h
Production: 36000/113287 | 9.64 image/s | ETA=2.23h
Production: 38000/113287 | 9.64 image/s | ETA=2.17h
Production: 400

## Cell 5 — Full-corpus validation và Parquet export

Kiểm đúng 113.287 ID theo thứ tự, BOS/EOS, độ dài, UNK, checkpoint provenance
và deterministic spot. Sau đó xuất Parquet nén ZSTD cho Phase 4.


In [5]:
CELL_STARTED_AT = time.perf_counter()

records = []
caption_set = set()
total_lexical = 0
total_unknown = 0
maximum_lexical_length = 0
with FINAL_JSONL.open("r", encoding="utf-8") as handle:
    for expected_index, line in enumerate(handle):
        record = json.loads(line)
        assert record["global_index"] == expected_index
        assert record["image_id"] == TRAIN_IMAGE_IDS[expected_index]
        assert record["caption"]
        assert record["teacher_checkpoint_sha256"] == (
            phase3b_ready["checkpoint_sha256"]
        )
        assert record["shard_id"] == 0
        student_ids = [int(value) for value in record["student_token_ids"]]
        assert student_ids[0] == 1
        assert student_ids[-1] == 2
        assert 2 <= len(student_ids) <= 32
        assert 0 not in student_ids
        lexical_count = len(student_ids) - 2
        total_lexical += lexical_count
        total_unknown += sum(value == 3 for value in student_ids)
        maximum_lexical_length = max(
            maximum_lexical_length, lexical_count
        )
        caption_set.add(record["caption"])
        records.append(record)
assert len(records) == 113287
assert len({record["image_id"] for record in records}) == 113287
assert set(record["image_id"] for record in records).isdisjoint(
    VALIDATION_IDS | TEST_IDS
)

# Deterministic spot-check of the first production batch.
spot_dataset = TrainImages(start_index=0)
spot_samples = [spot_dataset[index] for index in range(8)]
spot_images = torch.stack([sample["image"] for sample in spot_samples])
spot_teacher_ids, spot_captions = generate_batch(spot_images)
assert spot_teacher_ids == [
    record["teacher_token_ids"] for record in records[:8]
]
assert spot_captions == [
    record["caption"] for record in records[:8]
]

PARQUET_PATH = (
    PSEUDOCAPTION_DIR
    / "teacher_a_expansionnet_v2_train_restval.parquet"
)
table = pa.Table.from_pydict(
    {
        "global_index": [
            int(record["global_index"]) for record in records
        ],
        "image_id": [int(record["image_id"]) for record in records],
        "caption": [record["caption"] for record in records],
        "teacher_token_ids": [
            record["teacher_token_ids"] for record in records
        ],
        "student_token_ids": [
            record["student_token_ids"] for record in records
        ],
        "teacher_checkpoint_sha256": [
            record["teacher_checkpoint_sha256"] for record in records
        ],
        "shard_id": [0] * len(records),
    }
)
pq.write_table(
    table,
    PARQUET_PATH,
    compression="zstd",
    row_group_size=8192,
)
parquet_probe = pq.read_table(
    PARQUET_PATH,
    columns=["global_index", "image_id", "student_token_ids"],
)
assert parquet_probe.num_rows == 113287
assert parquet_probe.column("global_index")[0].as_py() == 0
assert parquet_probe.column("image_id")[0].as_py() == TRAIN_IMAGE_IDS[0]

unknown_rate = total_unknown / max(1, total_lexical)
SUMMARY_PATH = (
    PSEUDOCAPTION_DIR
    / "teacher_a_expansionnet_v2_pseudocaption_summary.json"
)
write_json(
    SUMMARY_PATH,
    {
        "schema_version": "phase3d_pseudocaption_summary_v1",
        "created_at_utc": now_utc(),
        "records": 113287,
        "unique_image_ids": 113287,
        "unique_captions": len(caption_set),
        "mean_lexical_tokens": total_lexical / 113287,
        "maximum_lexical_tokens": maximum_lexical_length,
        "unknown_tokens": total_unknown,
        "unknown_token_rate": unknown_rate,
        "student_bos_eos_valid": True,
        "validation_overlap": 0,
        "held_out_test_overlap": 0,
        "deterministic_spot_exact": True,
        "jsonl_sha256": sha256_file(FINAL_JSONL),
        "parquet_sha256": sha256_file(PARQUET_PATH),
        "test_set_status": "sealed",
    },
)
write_json(
    AUDIT_DIR / "phase3d_cell05_validation_and_export.json",
    {
        "audit_version": "phase3d_expansionnet_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "records": 113287,
        "unique_image_ids": 113287,
        "unique_captions": len(caption_set),
        "image_order_exact": True,
        "student_bos_eos_valid": True,
        "maximum_lexical_tokens": maximum_lexical_length,
        "unknown_tokens": total_unknown,
        "unknown_token_rate": unknown_rate,
        "validation_overlap": 0,
        "held_out_test_overlap": 0,
        "deterministic_spot_exact": True,
        "parquet_rows": parquet_probe.num_rows,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03D EXPANSIONNET V2 CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Records                   : 113287")
print("Unique image IDs          : 113287")
print("Unique captions           :", len(caption_set))
print(
    "Mean lexical tokens      :",
    round(total_lexical / 113287, 4),
)
print("Maximum lexical tokens    :", maximum_lexical_length)
print("Unknown tokens            :", total_unknown)
print("Unknown-token rate        :", f"{unknown_rate:.6%}")
print("BOS/EOS valid             : True")
print("Validation overlap        : 0")
print("Held-out Test overlap     : 0")
print("Deterministic spot exact  : True")
print("Parquet rows              : 113287")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 6 code cells")
print("Next                      : manifest and completion marker")


PHASE 03D EXPANSIONNET V2 CELL 5 STATUS
Status                    : PASSED
Records                   : 113287
Unique image IDs          : 113287
Unique captions           : 60899
Mean lexical tokens      : 9.5162
Maximum lexical tokens    : 17
Unknown tokens            : 0
Unknown-token rate        : 0.000000%
BOS/EOS valid             : True
Validation overlap        : 0
Held-out Test overlap     : 0
Deterministic spot exact  : True
Parquet rows              : 113287
Test Set status           : SEALED
Progress                  : 5 / 6 code cells
Next                      : manifest and completion marker


## Cell 6 — Manifest, completion marker và Phase 03E hand-off

Hash toàn bộ JSONL/Parquet/contracts/audits và chỉ tạo completion marker khi
mọi gate đều đạt.


In [6]:
CELL_STARTED_AT = time.perf_counter()

artifact_paths = sorted(
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
    and path.name not in {
        "phase3d_expansionnet_v2_pseudocaption_manifest.csv",
        "SHA256SUMS",
        "PHASE3_TEACHER_A_EXPANSIONNET_V2_"
        "PSEUDOCAPTIONS_COMPLETE.json",
    }
)
MANIFEST_CSV = (
    MANIFEST_DIR / "phase3d_expansionnet_v2_pseudocaption_manifest.csv"
)
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    for artifact in artifact_paths:
        writer.writerow(
            {
                "relative_path": artifact.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "size_bytes": artifact.stat().st_size,
                "sha256": sha256_file(artifact),
            }
        )
with MANIFEST_CSV.open(
    "r", encoding="utf-8", newline=""
) as handle:
    manifest_rows = list(csv.DictReader(handle))
SHA256SUMS = MANIFEST_DIR / "SHA256SUMS"
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

COMPLETE_PATH = (
    OUTPUT_ROOT
    / "PHASE3_TEACHER_A_EXPANSIONNET_V2_"
    "PSEUDOCAPTIONS_COMPLETE.json"
)
write_json(
    COMPLETE_PATH,
    {
        "schema_version": "phase3d_expansionnet_pseudocaptions_v1",
        "created_at_utc": now_utc(),
        "status": "complete",
        "phase": "03D",
        "teacher": "ExpansionNet v2",
        "matched_student": "S1 MobileNetV3-Large",
        "teacher_checkpoint_sha256": (
            phase3b_ready["checkpoint_sha256"]
        ),
        "records": 113287,
        "unique_image_ids": 113287,
        "selected_batch_size": 8,
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_decoder_steps": 31,
        "resume_used": RESUME_USED,
        "jsonl": {
            "relative_path": FINAL_JSONL.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "size_bytes": FINAL_JSONL.stat().st_size,
            "sha256": sha256_file(FINAL_JSONL),
        },
        "parquet": {
            "relative_path": PARQUET_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "size_bytes": PARQUET_PATH.stat().st_size,
            "sha256": sha256_file(PARQUET_PATH),
            "rows": 113287,
        },
        "summary": {
            "relative_path": SUMMARY_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(SUMMARY_PATH),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
                "rows": len(manifest_rows),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "direct_logit_kd_allowed": False,
        "sequence_kd_dataset_ready": True,
        "phase03e_handoff_authorized": True,
        "training_started": False,
        "test_parquet_opened": False,
        "test_caption_read": False,
        "test_images_read": False,
        "test_metrics_computed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 03D EXPANSIONNET V2 FINAL STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher                   : A — ExpansionNet v2")
print("Matched Student           : S1 — MobileNetV3-Large")
print("Pseudo-caption records    : 113287 / 113287")
print("JSONL / Parquet           : READY / READY")
print("Direct Logit KD           : PROHIBITED")
print("Sequence KD dataset       : READY")
print("Manifest rows             :", len(manifest_rows), "VERIFIED")
print("Training started          : False")
print("Test Set status           : SEALED")
print("COMPLETE marker           :", COMPLETE_PATH)
print("COMPLETE SHA-256          :", sha256_file(COMPLETE_PATH))
print("Progress                  : 6 / 6 code cells")
print(
    "Next                      : Phase 03E Teacher A replacement "
    "hand-off, then update Phase 4 A→S1 to Sequence KD"
)


PHASE 03D EXPANSIONNET V2 FINAL STATUS
Status                    : PASSED
Teacher                   : A — ExpansionNet v2
Matched Student           : S1 — MobileNetV3-Large
Pseudo-caption records    : 113287 / 113287
JSONL / Parquet           : READY / READY
Direct Logit KD           : PROHIBITED
Sequence KD dataset       : READY
Manifest rows             : 9 VERIFIED
Training started          : False
Test Set status           : SEALED
COMPLETE marker           : /kaggle/working//phase3_teacher_a_expansionnet_v2_pseudocaptions/PHASE3_TEACHER_A_EXPANSIONNET_V2_PSEUDOCAPTIONS_COMPLETE.json
COMPLETE SHA-256          : 5c1a0696698fb6b53c9612f7805dea2e42fbc3fb7635d0e63a4731dad4ddb101
Progress                  : 6 / 6 code cells
Next                      : Phase 03E Teacher A replacement hand-off, then update Phase 4 A→S1 to Sequence KD
